# Single test: the final circuits on FF++ test and Celeb-DF

The plan's one-time test. The finalists of `final_circuits.ipynb` (per backbone: random search, GA, PSO, memetic GA, first 8 PCA, PLS 8 and greedy; with and without a CZ ring; 5 seeds) are loaded from their saved weights, not retrained, and scored once on FF++ test and on the Celeb-DF v2 test list. The linear probe on all features is scored the same way as the ceiling.

**Two modes.**
- `run_test = False` (**check**, run first): loads every frozen weight file, recomputes its val AUC and stops the notebook if it differs from the value `final_circuits` recorded; refits the probe with `pool_prep`'s chosen C and checks its val AUC against the bottleneck table. FF++ test and Celeb-DF are not read.
- `run_test = True` (**the single test**, run once, after the check passes): the same checks, then FF++ test and Celeb-DF.

**The frozen list.** The finalists, seeds, entangler settings and weight files come from `final_circuits`' output and are written to `frozen_finalists.csv` with an MD5 of each weight file before anything is scored. Nothing is picked or tuned here.

**What is reported** (video AUC: a video's score is the mean of its frame probabilities; 95% bootstrap intervals over videos, 1,000 resamples):
- per finalist and entangler, the mean over the 5 seeds on FF++ test (all manipulations, and each manipulation's fakes against all real videos) and on Celeb-DF;
- the CZ ring minus no entangler, paired by seed;
- seed-averaged finalists (no entangler) against PLS 8 and against random search's finalist, as a paired bootstrap of the AUC difference;
- each finalist as a share of the linear probe's AUC (C3).

**Inputs (Add Input):**
- the output of `malakkhaled202300369/final-circuits` (folder `final_circuits/`: `finalists.csv`, `trainings.csv`, `weights/`)
- `malakkhaled202300369/nic-pool` (folder `pool/`: the pools, rows and the bottleneck table)
- raw features, for the probe: `malakkhaled202300369/nic-clip768-l14-dataset`, `malakkhaled202300369/nic-clip-dataset`, `malakkhaled202300369/efficientnet-features`

**Settings:** Accelerator None (CPU). Internet On, for the PennyLane install.

## 1. Settings

In [1]:
run_test = True            # False: check mode. True: the single test on FF++ test and Celeb-DF, run once
backbones = ["l14", "b16", "effb4"]
seeds = [0, 1, 2, 3, 4]
entanglers = ["none", "CZ ring"]
n_boot = 1000
n_jobs = 0                  # 0 = one job per CPU core
machine_name = "kaggle-farida"
seed = 0
out_dir = "/kaggle/working/single_test" if run_test else "/kaggle/working/single_test_check"

# raw features per backbone: (folder, file suffix), the same lookup as pool_prep
raw_features = {"b16": ("features", "_clip.npy"), "l14": ("features_l14", "_clip.npy"), "effb4": ("features", "_effb4.npy")}

## 2. Install and imports

In [2]:
!pip install -q pennylane==0.45.1

In [3]:
import os
# one thread per process, set before numpy and torch load
os.environ["OMP_NUM_THREADS"] = "1"
import glob
import json
import time
import hashlib
import platform
import multiprocessing as mp
import numpy as np
import pandas as pd
import sklearn
import torch
import pennylane as qml
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

torch.set_num_threads(1)
os.makedirs(out_dir, exist_ok=True)
if n_jobs == 0:
    n_jobs = os.cpu_count()
start_all = time.time()
print("MODE:", "SINGLE TEST (FF++ test and Celeb-DF)" if run_test else "check only (val)",
      "| backbones", backbones, "| jobs", n_jobs, "| pennylane", qml.__version__, "| torch", torch.__version__)

MODE: SINGLE TEST (FF++ test and Celeb-DF) | backbones ['l14', 'b16', 'effb4'] | jobs 4 | pennylane 0.45.1 | torch 2.11.0+cpu


## 3. Inputs and the frozen list

In [4]:
def find_all(filename):
    return sorted(glob.glob("/kaggle/input/**/" + filename, recursive=True))


def find_one(filename, folder=None):
    hits = find_all(filename)
    if folder is not None:
        hits = [p for p in hits if os.path.basename(os.path.dirname(p)) == folder]
    assert len(hits) == 1, (filename, folder, hits)
    return hits[0]


fc_dir = os.path.dirname(find_one("finalists.csv", "final_circuits"))
finalists = pd.read_csv(f"{fc_dir}/finalists.csv")
trainings = pd.read_csv(f"{fc_dir}/trainings.csv")
pool_dir = os.path.dirname(find_one("columns.json", "pool"))
bottleneck = pd.read_csv(f"{pool_dir}/bottleneck_table.csv")
print("finalists:", len(finalists), "| trainings recorded:", len(trainings))


def weight_file(b, key, entangler, s):
    # the naming rule of final_circuits
    name = f"{b}_{key}_{entangler}_{s}".replace("|", "_").replace(" ", "_").replace("+", "p")
    return f"{fc_dir}/weights/{name}.pt"


def md5(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


frozen = []
for _, p in finalists[finalists["backbone"].isin(backbones)].iterrows():
    for e in entanglers:
        for s in seeds:
            t = trainings[(trainings["backbone"] == p["backbone"]) & (trainings["key"] == p["key"])
                          & (trainings["entangler"] == e) & (trainings["seed"] == s)]
            assert len(t) == 1, (p["backbone"], p["finalist"], e, s, len(t))
            path = weight_file(p["backbone"], p["key"], e, s)
            assert os.path.exists(path), path
            frozen.append({"backbone": p["backbone"], "finalist": p["finalist"], "entangler": e, "seed": s,
                           "key": p["key"], "genome": t["genome"].iloc[0], "recorded val": t["val_auc"].iloc[0],
                           "weight file": os.path.basename(path), "md5": md5(path)})
frozen = pd.DataFrame(frozen)
frozen.to_csv(f"{out_dir}/frozen_finalists.csv", index=False)
print("frozen:", len(frozen), "circuits,", frozen.drop_duplicates(["backbone", "key", "entangler", "seed"]).shape[0],
      "distinct weight files")
print(finalists[finalists["backbone"].isin(backbones)].to_string(index=False))

finalists: 21 | trainings recorded: 340
frozen: 210 circuits, 200 distinct weight files
backbone    finalist                                                   key  picked_val_seed0
     l14      random      pca37.pca112.pls0.pls1.pls2.pls4.pls6.pls7|RX|RY          0.865536
     l14          GA       pca0.pls0.pls1.pls2.pls4.pls5.pls6.pls7|H+RZ|RY          0.891224
     l14         PSO     pca0.pca3.pls0.pls1.pls2.pls5.pls6.pls13|H+RZ|Rot          0.888163
     l14  memetic GA        pls0.pls1.pls2.pls3.pls4.pls5.pls6.pls11|RX|RY          0.893163
     l14 first 8 PCA        pca0.pca1.pca2.pca3.pca4.pca5.pca6.pca7|RY|Rot          0.736110
     l14       PLS 8         pls0.pls1.pls2.pls3.pls4.pls5.pls6.pls7|RX|RY          0.889094
     l14      greedy        pls0.pls1.pls2.pls3.pls4.pls5.pls6.pls11|RX|RY          0.893163
     b16      random      pca37.pca112.pls0.pls1.pls2.pls4.pls6.pls7|RX|RY          0.846977
     b16          GA      pls0.pls1.pls2.pls3.pls5.pls8.pls10.pls14|RX|Rot 

## 4. Splits

For each backbone, the pool angles and rows of val, FF++ test and Celeb-DF, with video ids, video labels and fixed bootstrap resamples. In check mode only val is loaded.

In [5]:
D = {}


def split_info(rows, rng):
    video = rows.groupby(["method", "video_id"]).ngroup().to_numpy()
    n = video.max() + 1
    video_y = np.zeros(n)
    video_y[video] = rows["label"].to_numpy()
    video_method = np.empty(n, dtype=object)
    video_method[video] = rows["method"].to_numpy()
    return {"video": video, "video_y": video_y, "video_method": video_method,
            "count": np.bincount(video, minlength=n), "boot": rng.integers(0, n, (n_boot, n))}


def load_backbone(b):
    D.clear()
    names = ["ffpp_val"] + (["ffpp_test", "celebdf_test"] if run_test else [])
    D["backbone"] = b
    D["x"], D["splits"] = {}, {}
    for name in names:
        rows = pd.read_csv(f"{pool_dir}/{name}_rows.csv", dtype={"video_id": str})
        x = np.load(f"{pool_dir}/{b}_{name}_pool.npy").astype(np.float64)
        assert len(x) == len(rows), (b, name)
        D["x"][name] = x
        D["splits"][name] = split_info(rows, np.random.default_rng(seed))
        s = D["splits"][name]
        print(f"{b} {name}: {len(rows)} frames, {len(s['video_y'])} videos ({int(s['video_y'].sum())} fake)")

## 5. The circuit

The `final_circuits` circuit, unchanged: encoding, a rotation layer, with the CZ ring a CZ ring and a second rotation layer, ⟨Z⟩ on every qubit, weighted sum plus bias. Weights are loaded, not trained.

In [6]:
n_qubits = 8
angles_per_rotation = {"RY": 1, "RX+RY": 2, "Rot": 3}


def rotation_layer(w, rotation):
    for i in range(n_qubits):
        if rotation == "RY":
            qml.RY(w[i, 0], wires=i)
        elif rotation == "RX+RY":
            qml.RX(w[i, 0], wires=i)
            qml.RY(w[i, 1], wires=i)
        else:
            qml.Rot(w[i, 0], w[i, 1], w[i, 2], wires=i)


def make_qnode(genome, entangler):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch", diff_method="backprop")
    def circuit(x, w):
        for i in range(n_qubits):
            if genome["encoding"] == "RX":
                qml.RX(x[:, i], wires=i)
            elif genome["encoding"] == "RY":
                qml.RY(x[:, i], wires=i)
            else:
                qml.Hadamard(wires=i)
                qml.RZ(x[:, i], wires=i)
        rotation_layer(w[0], genome["rotation"])
        if entangler == "CZ ring":
            for i in range(n_qubits):
                qml.CZ(wires=[i, (i + 1) % n_qubits])
            rotation_layer(w[1], genome["rotation"])
        return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

    return circuit


class VQC(torch.nn.Module):
    def __init__(self, genome, entangler):
        super().__init__()
        self.circuit = make_qnode(genome, entangler)
        layers = 2 if entangler == "CZ ring" else 1
        k = angles_per_rotation[genome["rotation"]]
        self.w = torch.nn.Parameter(0.1 * torch.randn(layers, n_qubits, k, dtype=torch.float64))
        self.readout = torch.nn.Linear(n_qubits, 1, dtype=torch.float64)

    def forward(self, x):
        z = torch.stack(self.circuit(x, self.w), dim=-1).to(torch.float64)
        return self.readout(z).squeeze(-1)

## 6. Scoring

`scores` turns frame probabilities into video scores; `metrics` gives the video AUC with its bootstrap interval and, on FF++ test, each manipulation's fakes against all real videos.

In [7]:
fake_methods = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]


def video_scores(name, prob):
    s = D["splits"][name]
    return np.bincount(s["video"], weights=prob, minlength=len(s["video_y"])) / s["count"]


def metrics(name, scores):
    s = D["splits"][name]
    auc = roc_auc_score(s["video_y"], scores)
    boot = [roc_auc_score(s["video_y"][i], scores[i]) for i in s["boot"]]
    out = {f"{name} auc": auc, f"{name} low": np.percentile(boot, 2.5), f"{name} high": np.percentile(boot, 97.5)}
    if name == "ffpp_test":
        for m in fake_methods:
            keep = (s["video_method"] == "original") | (s["video_method"] == m)
            out[f"ffpp_test {m}"] = roc_auc_score(s["video_y"][keep], scores[keep])
    return out


def score_circuit(item):
    torch.set_num_threads(1)
    genome = json.loads(item["genome"])
    model = VQC(genome, item["entangler"])
    model.load_state_dict(torch.load(f"{fc_dir}/weights/{item['weight file']}"))
    model.eval()
    row = {k: item[k] for k in ("backbone", "key", "entangler", "seed")}
    vs = {}
    with torch.no_grad():
        for name, x in D["x"].items():
            xt = torch.tensor(x[:, genome["picks"]], dtype=torch.float64)
            prob = torch.cat([torch.sigmoid(model(xt[i:i + 4096])) for i in range(0, len(xt), 4096)]).numpy()
            vs[name] = video_scores(name, prob)
            if name == "ffpp_val":
                row["val auc"] = roc_auc_score(D["splits"][name]["video_y"], vs[name])
            else:
                row.update(metrics(name, vs[name]))
    return row, vs

## 7. The linear probe on all features

`pool_prep`'s full linear probe: the feature vector scaled to length 1, standardized, logistic regression with the C `pool_prep` picked on the inner split, refit on all FF++ train frames. Its val AUC must match the bottleneck table.

In [8]:
def probe_scores(b):
    folder, suffix = raw_features[b]
    names = ["ffpp_train"] + list(D["x"])
    xn = {}
    for name in names:
        path = find_one(name + suffix, folder)
        raw_rows = pd.read_csv(os.path.join(os.path.dirname(path), name + "_rows.csv"), dtype={"video_id": str})
        pool_rows = pd.read_csv(f"{pool_dir}/{name}_rows.csv", dtype={"video_id": str})
        assert (raw_rows["path"].to_numpy() == pool_rows["path"].to_numpy()).all(), (b, name)
        x = np.load(path).astype(np.float64)
        xn[name] = x / np.linalg.norm(x, axis=1, keepdims=True)
    y = pd.read_csv(f"{pool_dir}/ffpp_train_rows.csv")["label"].to_numpy()
    row = bottleneck[(bottleneck["backbone"] == b) & (bottleneck["input"] == "full linear probe")].iloc[0]
    model = make_pipeline(StandardScaler(), LogisticRegression(C=float(row["C"]), max_iter=5000)).fit(xn["ffpp_train"], y)
    out = {"backbone": b, "C": float(row["C"]), "val in pool_prep": float(row["val"])}
    vs = {}
    for name in D["x"]:
        vs[name] = video_scores(name, model.predict_proba(xn[name])[:, 1])
        if name == "ffpp_val":
            out["val auc"] = roc_auc_score(D["splits"][name]["video_y"], vs[name])
        else:
            out.update(metrics(name, vs[name]))
    return out, vs

## 8. Run

Per backbone: score every distinct frozen weight file in parallel, then the probe. The val check runs first: a circuit whose recomputed val AUC differs from the recorded one by more than 1e-6, or a probe more than 1e-3 off the bottleneck table, stops the notebook.

In [9]:
circuit_rows, probe_rows = [], []
video_store = {}
for b in backbones:
    t0 = time.time()
    load_backbone(b)
    items = frozen[frozen["backbone"] == b].drop_duplicates(["key", "entangler", "seed"]).to_dict("records")
    with mp.get_context("fork").Pool(n_jobs) as workers:
        results = workers.map(score_circuit, items)
    for item, (row, vs) in zip(items, results):
        row["recorded val"] = item["recorded val"]
        circuit_rows.append(row)
        for name, v in vs.items():
            video_store[f"{b}|{item['key']}|{item['entangler']}|{item['seed']}|{name}"] = v
    gap = max(abs(r["val auc"] - r["recorded val"]) for r in circuit_rows if r["backbone"] == b)
    print(f"{b}: {len(items)} circuits scored, largest val difference from final_circuits {gap:.2e}")
    assert gap < 1e-6, "a loaded circuit does not reproduce its recorded val AUC"

    p, vs = probe_scores(b)
    probe_rows.append(p)
    for name, v in vs.items():
        video_store[f"{b}|probe|{name}"] = v
    print(f"{b}: probe val {p['val auc']:.4f} (pool_prep {p['val in pool_prep']:.4f}), {(time.time() - t0) / 60:.1f} min")
    assert abs(p["val auc"] - p["val in pool_prep"]) < 1e-3, "the probe does not reproduce pool_prep's val AUC"

scored = pd.DataFrame(circuit_rows)
probes = pd.DataFrame(probe_rows)
# map the scored weight files back onto every finalist role (one genome can stand for two finalists)
per_seed = frozen.drop(columns=["genome", "recorded val"]).merge(scored, on=["backbone", "key", "entangler", "seed"])
per_seed.to_csv(f"{out_dir}/per_seed.csv", index=False)
probes.to_csv(f"{out_dir}/probe.csv", index=False)
np.savez_compressed(f"{out_dir}/video_scores.npz", **{k.replace("|", "__").replace(" ", "_"): v for k, v in video_store.items()})
print("val check passed for every circuit and probe")
if not run_test:
    print("check mode: FF++ test and Celeb-DF were not read. Set run_test = True for the single test.")

l14 ffpp_val: 22396 frames, 700 videos (560 fake)
l14 ffpp_test: 22400 frames, 700 videos (560 fake)
l14 celebdf_test: 16572 frames, 518 videos (340 fake)
l14: 60 circuits scored, largest val difference from final_circuits 0.00e+00
l14: probe val 0.9049 (pool_prep 0.9049), 5.5 min
b16 ffpp_val: 22396 frames, 700 videos (560 fake)
b16 ffpp_test: 22400 frames, 700 videos (560 fake)
b16 celebdf_test: 16572 frames, 518 videos (340 fake)
b16: 70 circuits scored, largest val difference from final_circuits 0.00e+00
b16: probe val 0.8896 (pool_prep 0.8896), 5.9 min
effb4 ffpp_val: 22396 frames, 700 videos (560 fake)
effb4 ffpp_test: 22400 frames, 700 videos (560 fake)
effb4 celebdf_test: 16572 frames, 518 videos (340 fake)
effb4: 70 circuits scored, largest val difference from final_circuits 0.00e+00
effb4: probe val 0.7563 (pool_prep 0.7563), 7.0 min
val check passed for every circuit and probe


## 9. Test results per finalist

Mean over the 5 seeds, the seed standard deviation, and the mean 95% interval, for FF++ test (with each manipulation) and Celeb-DF, next to the probe.

In [10]:
if run_test:
    cols = ["ffpp_test auc", "ffpp_test low", "ffpp_test high"] + [f"ffpp_test {m}" for m in fake_methods] \
           + ["celebdf_test auc", "celebdf_test low", "celebdf_test high"]
    summary = per_seed.groupby(["backbone", "finalist", "entangler"], sort=False).agg(
        **{"val": ("val auc", "mean")}, **{c: (c, "mean") for c in cols},
        **{"ffpp_test sd": ("ffpp_test auc", "std"), "celebdf_test sd": ("celebdf_test auc", "std")}).reset_index()
    summary.to_csv(f"{out_dir}/test_summary.csv", index=False)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    for b in backbones:
        print(f"\n{b}")
        print(summary[summary["backbone"] == b].drop(columns=["backbone"]).round(4).to_string(index=False))
    print("\nlinear probe on all features")
    print(probes.round(4).to_string(index=False))


l14
   finalist entangler    val  ffpp_test auc  ffpp_test low  ffpp_test high  ffpp_test Deepfakes  ffpp_test Face2Face  ffpp_test FaceSwap  ffpp_test NeuralTextures  celebdf_test auc  celebdf_test low  celebdf_test high  ffpp_test sd  celebdf_test sd
     random      none 0.8637         0.8679         0.8386          0.8948               0.9721               0.8528              0.9471                    0.6995            0.7687            0.7252             0.8117        0.0008           0.0038
     random   CZ ring 0.8638         0.8675         0.8383          0.8943               0.9719               0.8527              0.9470                    0.6985            0.7679            0.7248             0.8109        0.0010           0.0044
         GA      none 0.8899         0.8930         0.8664          0.9176               0.9732               0.8819              0.9557                    0.7610            0.7758            0.7326             0.8165        0.0008           0.0036

## 10. CZ ring against no entangler on test

Paired by seed: the ring's AUC minus the no-entangler AUC of the same finalist and seed.

In [11]:
if run_test:
    ring = []
    for (b, f), g in per_seed.groupby(["backbone", "finalist"], sort=False):
        for name in ["ffpp_test auc", "celebdf_test auc"]:
            w = g.pivot(index="seed", columns="entangler", values=name)
            d = w["CZ ring"] - w["none"]
            ring.append({"backbone": b, "finalist": f, "split": name.replace(" auc", ""), "ring - none mean": d.mean(),
                         "min": d.min(), "max": d.max(), "seeds where the ring helps": int((d > 0).sum())})
    ring = pd.DataFrame(ring)
    ring.to_csv(f"{out_dir}/entangler_test.csv", index=False)
    print(ring.round(4).to_string(index=False))

backbone    finalist        split  ring - none mean     min     max  seeds where the ring helps
     l14      random    ffpp_test           -0.0003 -0.0008  0.0002                           1
     l14      random celebdf_test           -0.0008 -0.0033  0.0023                           2
     l14          GA    ffpp_test            0.0010  0.0008  0.0011                           5
     l14          GA celebdf_test            0.0004 -0.0006  0.0012                           3
     l14         PSO    ffpp_test            0.0013  0.0003  0.0028                           5
     l14         PSO celebdf_test            0.0029 -0.0081  0.0080                           4
     l14  memetic GA    ffpp_test           -0.0000 -0.0003  0.0001                           3
     l14  memetic GA celebdf_test           -0.0004 -0.0017  0.0004                           1
     l14 first 8 PCA    ffpp_test            0.0025 -0.0010  0.0053                           4
     l14 first 8 PCA celebdf_test       

## 11. Finalists against PLS 8 and random search, and as a share of the probe

Each finalist without an entangler, its video scores averaged over the 5 seeds. The AUC difference against PLS 8 and against random search's finalist is bootstrapped over the same resampled videos (paired), on FF++ test and Celeb-DF. The share of the probe is the finalist's AUC divided by the probe's.

In [12]:
if run_test:
    paired = []
    for b in backbones:
        load_backbone(b)
        f = finalists[finalists["backbone"] == b].set_index("finalist")["key"]

        def mean_scores(key, name):
            return np.mean([video_store[f"{b}|{key}|none|{s}|{name}"] for s in seeds], axis=0)

        for name in ["ffpp_test", "celebdf_test"]:
            s = D["splits"][name]
            probe_auc = roc_auc_score(s["video_y"], video_store[f"{b}|probe|{name}"])
            for fin, key in f.items():
                a = mean_scores(key, name)
                row = {"backbone": b, "split": name, "finalist": fin, "auc (seed mean scores)": roc_auc_score(s["video_y"], a),
                       "share of probe": roc_auc_score(s["video_y"], a) / probe_auc}
                for ref in ["PLS 8", "random"]:
                    r = mean_scores(f[ref], name)
                    diff = roc_auc_score(s["video_y"], a) - roc_auc_score(s["video_y"], r)
                    boot = [roc_auc_score(s["video_y"][i], a[i]) - roc_auc_score(s["video_y"][i], r[i]) for i in s["boot"]]
                    row[f"minus {ref}"] = diff
                    row[f"minus {ref} low"] = np.percentile(boot, 2.5)
                    row[f"minus {ref} high"] = np.percentile(boot, 97.5)
                paired.append(row)
    paired = pd.DataFrame(paired)
    paired.to_csv(f"{out_dir}/paired_differences.csv", index=False)
    print(paired.round(4).to_string(index=False))

l14 ffpp_val: 22396 frames, 700 videos (560 fake)
l14 ffpp_test: 22400 frames, 700 videos (560 fake)
l14 celebdf_test: 16572 frames, 518 videos (340 fake)
b16 ffpp_val: 22396 frames, 700 videos (560 fake)
b16 ffpp_test: 22400 frames, 700 videos (560 fake)
b16 celebdf_test: 16572 frames, 518 videos (340 fake)
effb4 ffpp_val: 22396 frames, 700 videos (560 fake)
effb4 ffpp_test: 22400 frames, 700 videos (560 fake)
effb4 celebdf_test: 16572 frames, 518 videos (340 fake)
backbone        split    finalist  auc (seed mean scores)  share of probe  minus PLS 8  minus PLS 8 low  minus PLS 8 high  minus random  minus random low  minus random high
     l14    ffpp_test      random                  0.8679          0.9533      -0.0259          -0.0398           -0.0115        0.0000            0.0000             0.0000
     l14    ffpp_test          GA                  0.8929          0.9808      -0.0008          -0.0059            0.0040        0.0251            0.0106             0.0395
     l14  

## 12. Run log

In [13]:
info = {"finished": time.strftime("%Y-%m-%d %H:%M:%S"), "machine": machine_name, "run_test": run_test,
        "backbones": backbones, "seeds": seeds, "entanglers": entanglers, "n_boot": n_boot, "n_jobs": n_jobs,
        "frozen circuits": int(len(frozen)), "total_minutes": round((time.time() - start_all) / 60, 1),
        "python": platform.python_version(), "numpy": np.__version__, "sklearn": sklearn.__version__,
        "torch": torch.__version__, "pennylane": qml.__version__}
with open(f"{out_dir}/runs.jsonl", "a") as f:
    f.write(json.dumps(info) + "\n")
!ls -lh {out_dir}

total 3.2M
-rw-r--r-- 1 root root 3.9K Nov 26 20:12 entangler_test.csv
-rw-r--r-- 1 root root  59K Nov 26 19:53 frozen_finalists.csv
-rw-r--r-- 1 root root 7.3K Nov 26 20:16 paired_differences.csv
-rw-r--r-- 1 root root  81K Nov 26 20:12 per_seed.csv
-rw-r--r-- 1 root root  925 Nov 26 20:12 probe.csv
-rw-r--r-- 1 root root  362 Nov 26 20:16 runs.jsonl
-rw-r--r-- 1 root root  12K Nov 26 20:12 test_summary.csv
-rw-r--r-- 1 root root 3.1M Nov 26 20:12 video_scores.npz
